# Week 5 - Function Approximation and Deep Q-Networks - Workbook

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Make a batch-shape bug visible. Start with **Exercises 1, 2 and 6** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-make-a-batch-shape-bug-visible).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, torch, matplotlib.pyplot as plt
import _pkg_loader  # registers w05_project
from w05_project import agent as agent_mod, train as train_mod, solutions
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## 1. Gather $Q(s,a)$ from a batched Q-network (one star)
Given `q_all` of shape `(B, A)` and integer `actions` of shape `(B,)`, return $Q(s_i, a_i)$ of shape `(B,)`.

> **Hint**: `q_all.gather(1, actions.long().unsqueeze(1)).squeeze(1)`.

In [ ]:
def gather_q(q_all, actions):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
q_all = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
out = select_exercise(solutions.exercise_1_gather_q, gather_q)(q_all, torch.tensor([2, 0]))
assert torch.allclose(out, torch.tensor([3.0, 4.0]))
print('exercise 1 OK')

## 2. DQN target with done-mask (two stars)
Compute $y_i = r_i + (1-d_i)\gamma \max_{a'} Q_{\theta^-}(s'_i, a')$.

> **Hint**: the done mask should zero out the bootstrap when the episode truly terminated.

In [ ]:
def dqn_target(reward, done, q_target_next, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
r = torch.tensor([1.0, 2.0])
d = torch.tensor([0.0, 1.0])
qn = torch.tensor([[0., 10.], [-1., 5.]])
assert torch.allclose(select_exercise(solutions.exercise_2_dqn_target, dqn_target)(r, d, qn, 0.9), torch.tensor([10.0, 2.0]))
print('exercise 2 OK')

## 3. Polyak soft update (one star)
$\theta^- \leftarrow (1-\tau)\theta^- + \tau \theta$ in place over all parameters.

In [ ]:
def polyak(target, online, tau):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
a_net = torch.nn.Linear(3, 1, bias=False); b_net = torch.nn.Linear(3, 1, bias=False)
a_net.weight.data.fill_(1.0); b_net.weight.data.fill_(5.0)
select_exercise(solutions.exercise_3_polyak, polyak)(b_net, a_net, 0.5)
assert torch.allclose(b_net.weight, torch.full_like(b_net.weight, 3.0))
print('exercise 3 OK')

## 4. $\varepsilon$-greedy from a Q-vector (one star)
Given `q_values` of shape `(A,)`, with probability $\varepsilon$ return a uniform random action; otherwise return $\arg\max$.

In [ ]:
def eps_greedy(q_values, eps, rng):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
q = np.array([1.0, 5.0, 2.0])
for _ in range(20):
    assert select_exercise(solutions.exercise_4_eps_greedy, eps_greedy)(q, 0.0, rng) == 1
print('exercise 4 OK')

## 5. Per-element TD error for PER (two stars)
Compute `|target - q_sa|` and detach. Used as the priority for Prioritized Experience Replay in Week 6.

In [ ]:
def td_error(q_sa, target):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
q = torch.tensor([1., 2.]); t = torch.tensor([3., 0.])
assert torch.allclose(select_exercise(solutions.exercise_5_td_error, td_error)(q, t), torch.tensor([2., 2.]))
print('exercise 5 OK')

## 6. Huber loss (one star)
Use `torch.nn.functional.smooth_l1_loss(pred, target, reduction='mean', beta=delta)`.

In [ ]:
def huber(pred, target, delta=1.0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
p = torch.tensor([1., 2.]); t = torch.tensor([1., 2.])
assert select_exercise(solutions.exercise_6_huber, huber)(p, t).item() == 0.0
print('exercise 6 OK')
# A unit residual of 2 is in Huber's linear branch: 1*(2-.5)=1.5.
assert abs(float(select_exercise(solutions.exercise_6_huber, huber)(torch.tensor([0.]), torch.tensor([2.]), 1.0))-1.5)<1e-6


## 7. The deadly triad in action - Baird's counterexample (three stars)
Implement off-policy semi-gradient TD on a 7-state linear MDP where reward is 0 everywhere. Verify the parameter norm diverges.

> **Hint**: see `solutions.exercise_7_baird_divergence` for a clean implementation. The right way to see this is to log $\|w\|_2$ over iterations and plot on a log scale.

In [ ]:
norms = solutions.exercise_7_baird_divergence(n_iters=300, alpha=0.1)
plt.semilogy(norms); plt.xlabel('iter'); plt.ylabel('|w|'); plt.title('Baird divergence'); plt.show()
assert norms[-1] > 5 * norms[0]
print('exercise 7 OK')

## 8. Smoke training on LunarLander (two stars)
Train DQN for a few thousand steps; report the mean-of-last-10-episodes alongside a random-policy baseline. A short smoke run checks execution, not guaranteed learning success.

In [ ]:
log = solutions.exercise_8_smoke_dqn(n_steps=3000, seed=0)
mean_last = float(np.mean(log.episode_returns[-10:]))
print(f'mean last 10 = {mean_last:.1f}, episodes = {len(log.episode_returns)}, wall = {log.meta["wall_clock_s"]:.1f}s')
assert np.isfinite(mean_last)
print('Short-run return is a diagnostic, not a convergence guarantee.')
print('exercise 8 OK')

## 9. Action-value trace and matching application (three stars, meaningful objective)
**Matching angle.** Roll a trained policy out greedily and record $Q(s,\cdot)$ at every step. Then identify the state regions where the *gap* between $\max_a Q(s,a)$ and second-best is largest. These are states with strong **predicted action preference**. The gap to second-best is not the gap to a uniform policy (which uses the action mean), and learned Q gaps do not establish causal lift or calibrated certainty. Inspect actual rollout outcomes before interpreting them.

In [ ]:
import gymnasium as gym
log = solutions.exercise_8_smoke_dqn(n_steps=3000, seed=0)
# In practice you'd reuse the trained agent; for a self-contained exercise we re-train briefly here.
from w05_project.agent import DQNAgent, DQNConfig
cfg = DQNConfig(obs_dim=8, n_actions=4, eps_decay_steps=2000, warmup_steps=500, buffer_size=10000)
ag = DQNAgent(cfg, seed=0)
env = gym.make('LunarLander-v3'); s, _ = env.reset(seed=0)
for step in range(3000):
    a = ag.act(np.asarray(s, dtype=np.float32))
    s2, r, term, trunc, _ = env.step(a)
    ag.push(s, a, r, s2, term or trunc); ag.learn()
    s = s2 if not (term or trunc) else env.reset(seed=step + 1)[0]
states, qs, acts, rews = solutions.exercise_9_action_value_trace(ag, gym.make('LunarLander-v3'), max_steps=250, seed=0)
# action gap = best - second-best
sorted_q = np.sort(qs, axis=1)
gap = sorted_q[:, -1] - sorted_q[:, -2]
fig, ax = plt.subplots(figsize=(7, 3.0))
ax.plot(gap, color=RL_PALETTE['orange']); ax.set_xlabel('step'); ax.set_ylabel('action-value gap')
ax.set_title('Where the policy adds the most lift over uniform random')
fig.tight_layout(); plt.show()
assert np.isfinite(gap).all()
print('This trace uses an untrained action-value model; compare signs and scale without claiming measured lift.')
print('exercise 9 OK')

## Reflection
1. How does the target network mitigate moving-target feedback, and why does that not prove convergence?
2. What is the role of the replay buffer when the data is *already* i.i.d. (no temporal correlation between transitions)?
3. In the matching framing, what part of the LunarLander analysis transfers, and what does not?

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>1. Gather $Q(s,a)$ from a batched Q-network (one star)</summary>

> **Hint**: `q_all.gather(1, actions.long().unsqueeze(1)).squeeze(1)`.

</details>

<details>
<summary>2. DQN target with done-mask (two stars)</summary>

> **Hint**: the done mask should zero out the bootstrap when the episode truly terminated.

</details>

<details>
<summary>3. Polyak soft update (one star)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>5. Per-element TD error for PER (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>6. Huber loss (one star)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>7. The deadly triad in action - Baird&#x27;s counterexample (three stars)</summary>

> **Hint**: see `solutions.exercise_7_baird_divergence` for a clean implementation. The right way to see this is to log $\|w\|_2$ over iterations and plot on a log scale.

</details>

<details>
<summary>8. Smoke training on LunarLander (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>9. Action-value trace and matching application (three stars, meaningful objective)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — Replay reuse changes computation, not the number of episodes

**Goal.** Separate replay reuse, frozen-policy evaluation episodes and independent retraining. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** A frozen DQN has one evaluation return from each of 30 independent episodes.
Replay the same recorded returns eight times. Predict what happens to the
mean, the row-based standard error, and the episode-based standard error.
This controlled fixture separates reuse from genuinely new evaluation.

**Work a small case by hand.** For a fixed agent with population episode-return SD sigma=8, n=30
independent episodes give planning SE=8/sqrt(30)≈1.4606. Pretending eight
copies of each episode were new observations would use
8/sqrt(240)≈0.5164. The code estimates sigma from the observed sample and
uses a t critical value, so its numerical widths need not equal these known-SD
planning values. The 30 distinct episode IDs remain the actual units.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/05-resampling-dependence.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
from common.statistics import independent_mean_interval, cluster_means
rng_stats=np.random.default_rng(105)
episode_returns=rng_stats.normal(20,8,30)
reused=np.repeat(episode_returns,8); ids=np.repeat(np.arange(30),8)
real=independent_mean_interval(episode_returns,unit='independent evaluation episode')
false=independent_mean_interval(reused,unit='replayed row (invalid independence)')
deduplicated=independent_mean_interval(cluster_means(reused,ids),unit='independent evaluation episode')
assert np.isclose(real['standard_error'],deduplicated['standard_error'])
print('30 episode SE:',real['standard_error'],'240 reused-row SE (invalid):',false['standard_error'])
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
axes[0].plot(episode_returns,'o-',color=RL_PALETTE['blue'])
axes[0].set(xlabel='Independent frozen-policy evaluation episode',ylabel='Synthetic return')
for i,(label,r) in enumerate([('Original episodes',real),('Replayed rows',false),('Grouped episodes',deduplicated)]):
    axes[1].errorbar(i,r['estimate'],yerr=[[r['estimate']-r['low']],[r['high']-r['estimate']]],fmt='o',capsize=4)
axes[1].set_xticks([0,1,2],['Original','Replay IID (invalid)','Grouped'])
axes[1].set(ylabel='Mean return with 95% t interval')
fig.suptitle('More gradient updates do not create independent evaluation data')
fig.tight_layout();plt.show()


**Interpret and transfer.** Grouping recovers the original inferential sample size. Real DQN training
transitions are dependent and the policy changes over training; this example
does not justify applying an episode interval to that changing process.

**Transfer.** Add genuinely new frozen-policy episodes instead of duplicates.
Then separate uncertainty within one trained seed from variation across
independently trained seeds.


**Read your result.** Check that the original and grouped intervals coincide exactly. The replay-IID
interval is marked invalid: duplication preserves the estimate but does not
supply the independence premise needed for its narrower width.

<details><summary>Worked feedback for the changed assumption</summary>

Adding 210 genuinely independent evaluations of the same frozen agent gives
240 episodes and a planning SE smaller by sqrt(8). Copying the existing data
does not. Neither procedure samples new training runs, so it cannot measure
variation caused by independently retraining DQN.

</details>
